# Notebook 02 — Returns, Risk and Descriptive Analytics

**FIN 4600 · Lab 3 · Financial Data Analytics**

Duran, *Financial Services Technology* (3rd ed.), **Chapter 6 — Data Analytics**

---

Chapter 6 separates analytics into **descriptive** (what happened),
**predictive** (what is likely to happen) and **prescriptive** (what should we
do). This notebook is descriptive analytics: turning a price file into the
risk and performance measures that actually appear on a desk.

You already know these measures from FIN 4000 and FIN 4500. The point here is
not the formula — it is computing it correctly, at scale, from raw data.

**What you will build**

1. Simple and log returns, and why the distinction matters
2. Cumulative growth of a dollar
3. Annualised return, volatility and the Sharpe ratio
4. Rolling volatility, and realised vs implied (VIX)
5. Drawdown
6. The correlation matrix, and what it says about diversification
7. The return distribution, fat tails, and historical Value at Risk
8. Beta, computed two ways

**How to use this notebook.** Every line of code is already written and
commented. Run the cells top to bottom (`Shift + Enter`), read the markdown
before each cell, and check that what you see in the output matches what the
text tells you to look for. The three exercises have worked solutions. A Canvas
comprehension quiz on this notebook follows.

Notebook 01 introduced pandas, NumPy and matplotlib. Here they get a short
recap; the new material is the finance and statistics, plus **SciPy**, which
appears for the first time in section 7.

## 0. Setup

The next cell imports the libraries, locates the repository folder and sets a
few display options. It prints nothing; it is working if it runs without an
error.

**Quick recap of the toolkit (from Notebook 01).**

- **pandas** is the standard Python library for tables (`DataFrame`) and
  labelled columns (`Series`). Banks, asset managers and data vendors use it
  because a time-indexed table with aligned arithmetic is exactly what market
  data is.
- **NumPy** is the numerical engine underneath pandas: fast arrays and maths
  functions (`np.log`, `np.sqrt`, linear algebra). pandas columns are NumPy
  arrays with labels attached.
- **matplotlib** is Python's original plotting library. `plt.subplots()`
  returns a *figure* (the whole canvas) and an *axes* object `ax` (one chart
  inside it); every drawing command in this notebook is a method on `ax`.
- **`pathlib.Path`** builds file paths that work on Windows and macOS alike;
  `/` joins path pieces.

**New here:** `LinearSegmentedColormap` from matplotlib. A *colour map*
translates a number into a colour; this class builds one by blending
smoothly between a list of colours you choose. It is used for the correlation
heatmap in section 6.

`TRADING_DAYS = 252` is the market convention for the number of trading days
in a year (365 days, less weekends and about nine exchange holidays). Every
annualisation in this notebook uses it.

In [ ]:
# Path is pandas-free file-path handling from the standard library
from pathlib import Path

# NumPy: fast numerical arrays and maths functions (np.log, np.sqrt, linear algebra)
import numpy as np
# pandas: labelled tables (DataFrame) and columns (Series), the workhorse for market data
import pandas as pd
# pyplot is matplotlib's plotting interface; "plt" is the universal nickname
import matplotlib.pyplot as plt
# LinearSegmentedColormap builds a custom colour scale by blending a list of colours
from matplotlib.colors import LinearSegmentedColormap


# Walk up from the current folder until we find the one holding data/sp500_prices.csv
def find_repo_root(start=None):
    # The docstring below is the function's built-in help text
    """Return the repository root — the folder that contains data/sp500_prices.csv."""
    # Start from the given folder, or the notebook's working folder; resolve() makes it absolute
    here = Path(start or Path.cwd()).resolve()
    # Try this folder first, then each parent folder in turn, up to the drive root
    for candidate in [here, *here.parents]:
        # If the data file exists below this candidate, this candidate is the repo root
        if (candidate / "data" / "sp500_prices.csv").exists():
            # Stop searching and hand the folder back
            return candidate
    # Reaching here means no folder matched: stop with a helpful message instead of a vague error
    raise FileNotFoundError(
        "Could not find the repository root. In VS Code use File > Open Folder "  # message, part 1
        "and open the mtu4600-lab3-analytics folder itself, then re-run."  # message, part 2 (Python joins adjacent strings)
    )


# The repository root folder, found automatically so the notebook runs on any machine
REPO = find_repo_root()
# The data folder; "/" on a Path joins folder names safely on Windows and macOS
DATA = REPO / "data"
# Apply the course's chart style file (fonts, colours, grid) to every chart below
plt.style.use(REPO / "fin4600.mplstyle")

# Show up to 30 columns when a DataFrame is printed, instead of hiding the middle ones
pd.set_option("display.max_columns", 30)
# Let printed tables run to 130 characters wide before wrapping
pd.set_option("display.width", 130)

# The conventional number of trading days in a year; used to annualise daily figures
TRADING_DAYS = 252  # the conventional number of trading days in a year

### Load the prices and reshape them wide

The price file is stored **long**: one row per (date, ticker). Return and risk
maths needs it **wide**: one row per date, one column per ticker, so that a
single vectorised operation computes all 30 stocks at once. `pivot` does that
reshaping (you met it in Notebook 01).

We also build `sector_of`, a plain Python dictionary such as
`{"AAPL": "Information Technology", ...}`, used later to label and group
stocks by sector.

**Look for:** 1,259 trading days by 30 tickers, February 2013 to February
2018 — five years of a mostly rising market.

In [ ]:
# Rebuild the wide price matrix, so this notebook stands on its own
# Read the long price file; parse_dates turns the "date" text column into real datetimes
prices = pd.read_csv(DATA / "sp500_prices.csv", parse_dates=["date"])
# Read the company reference table (ticker, name, sector, ...)
companies = pd.read_csv(DATA / "sp500_companies.csv")

# Reshape long -> wide: rows = dates, columns = tickers, cells = closing price
close = prices.pivot(index="date", columns="ticker", values="close")
# Make "ticker" the index, keep the sector column, and convert to a {ticker: sector} dictionary
sector_of = companies.set_index("ticker")["sector"].to_dict()

# shape is (rows, columns): number of trading days by number of tickers
print(f"{close.shape[0]} trading days x {close.shape[1]} tickers")
# First and last dates in the index; .date() drops the time-of-day part for display
print(f"{close.index.min().date()} to {close.index.max().date()}")

## 1. Simple returns vs log returns

**Simple return** $R_t = \dfrac{P_t}{P_{t-1}} - 1$. This is what you actually
earn. Simple returns add up *across assets* — a portfolio's return is the
weighted sum of its holdings' simple returns.

**Log return** $r_t = \ln\!\left(\dfrac{P_t}{P_{t-1}}\right)$. Log returns add
up *across time* — a two-day log return is just the sum of the two daily log
returns. They are also closer to symmetric, which most statistical models
assume.

**Use simple returns for portfolio arithmetic; use log returns for
time-series modelling.** Mixing them up is a classic error, and for daily
equity data it is a small one — the two are nearly identical because daily
moves are small. For a stock that halves, it is a very large one.

#### Background: simple and log returns

**Intuition.** A price series is not what an investor cares about; the
*percentage change* is. Prices of different stocks are not comparable
(\$30 vs \$1,000), but returns are.

**The two definitions and how they relate.**

$$R_t = \frac{P_t}{P_{t-1}} - 1, \qquad r_t = \ln\frac{P_t}{P_{t-1}} = \ln(1 + R_t), \qquad R_t = e^{r_t} - 1$$

- **Across time, log returns add.** Over $n$ days the gross return is the
  product $\prod (1+R_t)$. Taking logs turns a product into a sum:
  $\ln \prod(1+R_t) = \sum r_t$. That is why time-series models, which love
  sums and averages, are built on log returns.
- **Across assets, simple returns add.** A portfolio with weights $w_i$ earns
  $R_p = \sum_i w_i R_i$ exactly. The same is *not* true for log returns,
  because the log of a weighted sum is not the weighted sum of logs.
- **Size of the gap.** A Taylor expansion gives
  $\ln(1+R) \approx R - R^2/2$, so the log return is always a little below the
  simple return, and the gap grows with the square of the move. At 1% the gap
  is 0.005 percentage points; at −50% the log return is −69%.
- **Symmetry.** A +50% then −50% move leaves you at 0.75 of where you started.
  In simple returns that looks like "+50, −50, net zero", which is false. In
  log returns it is $+0.405 - 0.693 = -0.288$, and $e^{-0.288} = 0.75$.
  Log returns also cannot go below −100% in a misleading way: a simple return
  is bounded at −1, a log return runs to $-\infty$, which makes them closer to
  symmetric and better suited to the normal-distribution models in section 7.

**How the next cell computes them.**

- `close.pct_change()` divides each row by the row above and subtracts 1 —
  the simple return, for all 30 columns at once. The first row is `NaN`
  because there is no previous price.
- `close.shift(1)` moves every price down one row, so that on each date the
  value is *yesterday's* price. `np.log(close / close.shift(1))` is then the
  log return. `shift` is the key pandas tool for "the previous observation"
  and reappears in Notebook 03, where shifting the wrong way causes
  look-ahead bias.

**Look for:** on a typical AAPL day the two returns agree to the fourth
decimal place; the largest gap over five years is only about 0.0034 (a third
of a percentage point).

In [ ]:
# Simple return for every stock and day: P_t / P_(t-1) - 1; the first row is NaN
simple_ret = close.pct_change()
# Log return: ln(P_t / P_(t-1)); shift(1) lines each price up with the previous day's price
log_ret = np.log(close / close.shift(1))

# Put AAPL's two return series side by side in one table, for comparison
comparison = pd.DataFrame(
    {"simple": simple_ret["AAPL"], "log": log_ret["AAPL"]}  # column name -> Series; pandas aligns them on date
).dropna()  # drop the first row, which is NaN because day one has no previous price

# Heading line for the printout
print("Daily returns are nearly identical:")
# First three days, rounded to 6 decimals so the small differences are visible
print(comparison.head(3).round(6))
# Largest absolute gap between simple and log return over all ~1,258 AAPL days
print(f"\nLargest absolute gap across all AAPL days: {(comparison['simple'] - comparison['log']).abs().max():.5f}")

### The gap grows with the size of the move

Daily moves are small, so the two return measures look interchangeable. The
next cell shows what happens when they are not small. It uses `np.log1p(x)`,
which computes $\ln(1 + x)$ — the conversion from a simple return to a log
return. (`log1p` exists because it is more accurate than `np.log(1 + x)` when
$x$ is tiny.)

**Look for:** the difference is about zero at 1%, −0.0945 at +50% and −1.4026
at −90%. A stock that loses 90% has a log return of −230%. Monthly or annual
returns, and anything in a crisis, are where mixing the two up really costs
you.

In [ ]:
# The gap grows with the size of the move
# A small table of hypothetical simple returns, from +1% to -90%
demo = pd.DataFrame({"simple_return": [0.01, 0.05, 0.10, 0.50, -0.50, -0.90]})
# Convert each to a log return: log1p(x) = ln(1 + x)
demo["log_return"] = np.log1p(demo["simple_return"])
# Log minus simple: always zero or negative, and larger for bigger moves
demo["difference"] = demo["log_return"] - demo["simple_return"]
# Display the table rounded to 4 decimals (the last expression in a cell is displayed)
demo.round(4)

### Why log returns add across time

The next cell checks the "log returns add across time" claim on real AAPL data,
using two-day windows.

- `rolling(2)` creates a sliding window of the current and previous row.
  `.apply(np.prod, raw=True)` multiplies the two values in each window
  (`raw=True` passes a plain NumPy array, which is faster). So
  `(1 + R).rolling(2).apply(np.prod) - 1` is the compounded two-day simple
  return: $(1+R_{t-1})(1+R_t) - 1$.
- `log_ret.rolling(2).sum()` simply **adds** the two daily log returns.
- `np.expm1(x)` computes $e^x - 1$, converting a log return back to a simple
  one (it is the inverse of `log1p`).

**Look for:** the first and third columns are identical on every row. Summing
logs and converting back gives exactly the compounded simple return; summing
*simple* returns would not.

In [ ]:
# Two-day compounded simple return: multiply (1 + R) over each 2-day window, then subtract 1
two_day_simple = (1 + simple_ret["AAPL"]).rolling(2).apply(np.prod, raw=True) - 1
# Two-day log return: just add the two daily log returns
two_day_log = log_ret["AAPL"].rolling(2).sum()

# Gather the three versions into one table to compare row by row
check = pd.DataFrame(
    {  # start of the dictionary
        "compounded simple": two_day_simple,  # (1+R1)(1+R2) - 1
        "summed log": two_day_log,  # r1 + r2
        "summed log converted back": np.expm1(two_day_log),  # e^(r1 + r2) - 1, back in simple-return units
    }
).dropna()  # remove the first rows where a full 2-day window is not yet available
# Show the first four rows, rounded to 6 decimals
check.head(4).round(6)

## 2. Growth of a dollar

The single most useful performance chart. Start with \$1 and compound.

**The calculation.** The value of \$1 after $T$ days is the running product of
gross returns:

$$W_T = \prod_{t=1}^{T} (1 + R_t)$$

`cumprod()` is pandas' cumulative product down each column, so
`(1 + simple_ret.fillna(0)).cumprod()` is the wealth path of all 30 stocks at
once. `fillna(0)` replaces the first-row `NaN` with a 0% return, so every line
starts at exactly \$1.

**Plotting idioms used here (and reused throughout).**

- `fig, ax = plt.subplots(figsize=(width, height))` — one chart, size in inches.
- `ax.plot(x, y, color=..., linewidth=..., label=..., zorder=...)` draws a
  line. `zorder` controls stacking: higher numbers are drawn on top, so the
  highlighted lines sit above the grey background ones.
- `ax.axhline(1.0, linestyle="--")` draws a horizontal reference line across
  the chart (here, break-even).
- `ax.annotate(text, xy=(x, y))` writes text at a data coordinate — used for
  direct labels at the right-hand end of each line.
- `ax.set_xlim(...)` widens the x-axis by 330 days (`pd.Timedelta`) to make
  room for those labels.
- `plt.show()` renders the figure.

**Look for:** BA and MSFT end far above \$1; GE ends below it — a loss over a
five-year bull market.

In [ ]:
# Wealth path of $1 in every stock: cumulative product of (1 + daily return); NaN first day -> 0%
growth = (1 + simple_ret.fillna(0)).cumprod()

# The four stocks to draw in colour; everything else becomes grey context
highlight = ["BA", "MSFT", "JNJ", "GE"]

# One figure (fig) containing one chart (ax), 9.5 x 5 inches
fig, ax = plt.subplots(figsize=(9.5, 5))
# Loop over all 30 tickers (the column names of the price matrix)
for ticker in close.columns:
    # Only the non-highlighted ones in this first pass
    if ticker not in highlight:
        # Thin light-grey line, zorder=1 so it sits underneath the coloured lines
        ax.plot(growth.index, growth[ticker], color="#d9d8d4", linewidth=1, zorder=1)
# Second pass: the four highlighted stocks
for ticker in highlight:
    # Thicker line in the style's default colour cycle, labelled for the legend, drawn on top
    ax.plot(growth.index, growth[ticker], linewidth=2, label=ticker, zorder=2)

# Dashed break-even line at $1
ax.axhline(1.0, color="#52514e", linewidth=0.8, linestyle="--")
# Chart title
ax.set_title("Growth of $1 invested February 2013")
# y-axis label
ax.set_ylabel("Value of $1")
# Blank x-axis label (the dates speak for themselves)
ax.set_xlabel("")
# Legend in the top-left corner, built from the label= arguments above
ax.legend(loc="upper left")

# Direct labels at the right edge, so identity is never colour-alone
# The final date in the sample, where the labels will be anchored
last_date = growth.index[-1]
# One label per highlighted stock
for ticker in highlight:
    # Write text at a data point on the chart
    ax.annotate(
        f"  {ticker}  ${growth[ticker].iloc[-1]:.2f}",  # e.g. "  BA  $4.10": ticker and final value of $1
        xy=(last_date, growth[ticker].iloc[-1]),  # position: last date, final wealth value
        fontsize=9,  # small text
        va="center",  # vertically centre the text on the line's end point
        color="#52514e",  # dark grey
    )
# Extend the x-axis 330 days past the last date so the labels fit inside the chart
ax.set_xlim(growth.index[0], last_date + pd.Timedelta(days=330))
# Render the figure
plt.show()

The grey lines are the other 26 stocks. Showing them de-emphasised gives the
reader context — how the highlighted names did *relative to the field* —
without adding 26 colours nobody can tell apart. Any chart that needs more
than about eight distinguishable colours is the wrong chart.

## 3. Annualising: return, volatility, Sharpe

Daily numbers are annualised by convention:

- mean return: $\times 252$
- volatility: $\times \sqrt{252}$ (variance scales with time, so standard
  deviation scales with its square root)

The square root rule assumes returns are independent across days. They are
not, quite — there is mild autocorrelation and strong volatility clustering —
so treat annualised volatility as a convention, not a truth.

#### Background: annualising return and volatility

Daily figures are tiny and hard to compare with anything; desks quote
everything per year.

**Mean return.** If each day has expected return $\mu_d$, a year of 252 days
has about $252\,\mu_d$. So `simple_ret.mean() * 252`. (This is the
*arithmetic* annual mean. The *geometric* or compound annual growth rate,
which is what the growth-of-a-dollar chart shows, is lower whenever returns
vary; roughly, geometric $\approx$ arithmetic $- \sigma^2/2$.)

**Volatility.** If daily returns are independent with variance $\sigma_d^2$,
the variance of a sum of 252 of them is $252\,\sigma_d^2$ (variances of
independent variables add). Volatility is the standard deviation, the square
root of the variance:

$$\sigma_{\text{annual}} = \sqrt{252\,\sigma_d^2} = \sigma_d\sqrt{252} \approx 15.9\,\sigma_d$$

So a stock with 1.3% daily volatility has about 20.6% annual volatility. The
same rule converts any horizon: a 10-day volatility is $\sigma_d\sqrt{10}$ —
the "square root of time" rule that regulators used for 10-day VaR.

`DataFrame.std()` computes the *sample* standard deviation (it divides by
$n-1$, not $n$), separately for each column.

#### Background: the Sharpe ratio

The Sharpe ratio is excess return per unit of total risk:

$$\text{Sharpe} = \frac{\bar R_{\text{annual}} - r_f}{\sigma_{\text{annual}}}$$

The numerator is what you earned *above* a riskless alternative; the
denominator is how bumpy the ride was. It lets you compare a quiet utility and
a volatile bank on one scale. A Sharpe of 1 over a long period is very good
for a single stock; most diversified equity funds run at 0.3 to 0.6 over a
full cycle.

Weaknesses worth knowing: it treats upside and downside volatility as equally
bad; it assumes volatility captures risk (it misses fat tails and drawdowns —
sections 5 and 7); and, as Exercise 1 shows, it is very noisy over short
windows.

**The next cell** chooses $r_f$. It filters the monthly 10-year Treasury file
to the sample window with a boolean mask (`(cond1) & (cond2)`; pandas needs
`&`, not `and`, and the brackets) and averages it. **Look for:** about 2.26%.

In [ ]:
# Risk-free rate. We only have the 10-year Treasury yield in this repo, so we
# use it as a stand-in. A correct Sharpe ratio uses a SHORT rate (3-month
# T-bills). This approximation overstates the risk-free rate and therefore
# understates every Sharpe ratio below. Flagging an approximation is part of
# the job.
# Monthly 10-year Treasury yields (in percent), with dates parsed as datetimes
us10y = pd.read_csv(DATA / "us10y_monthly.csv", parse_dates=["date"])
# Keep only months inside our price sample; each comparison gives True/False, & requires both
sample_window = us10y[(us10y["date"] >= "2013-02-01") & (us10y["date"] <= "2018-02-28")]
# Average yield over the window, divided by 100 to turn e.g. 2.26 (percent) into 0.0226 (decimal)
rf_annual = sample_window["yield_pct"].mean() / 100

# ":.2%" formats a decimal as a percentage with 2 decimal places
print(f"Average 10-year Treasury yield over the sample: {rf_annual:.2%}")
# Reminder that this is a proxy
print("(Used below as a proxy for the risk-free rate — see the note in the code.)")

### The risk-return table

The next cell builds `stats`, one row per stock, using the formulas above:

- `simple_ret.mean()` averages **each column** (each stock) over all days and
  returns a Series indexed by ticker; `* TRADING_DAYS` annualises it.
- `simple_ret.std() * np.sqrt(TRADING_DAYS)` is annualised volatility.
- Arithmetic between whole columns (`stats["ann_return"] - rf_annual`) works
  element by element — vectorisation, with no loop.
- `stats.index.map(sector_of)` looks each ticker up in the sector dictionary.

**Look for:** BA tops the table with a Sharpe of about 1.42 (33% a year at
21% volatility); GE is last at about −0.43, the only stock with a clearly
negative annual return. Note also how the Consumer Staples and Utilities names
(KO, PG, DUK) cluster near zero: low risk *and* low return in this period.

In [ ]:
# One row per ticker: annualised mean return and annualised volatility
stats = pd.DataFrame(
    {  # start of the dictionary
        "ann_return": simple_ret.mean() * TRADING_DAYS,  # mean daily simple return x 252
        "ann_volatility": simple_ret.std() * np.sqrt(TRADING_DAYS),  # daily sample std x sqrt(252)
    }
)
# Sharpe ratio: (annual return - risk-free rate) / annual volatility, for all stocks at once
stats["sharpe"] = (stats["ann_return"] - rf_annual) / stats["ann_volatility"]
# Look up each ticker's sector in the {ticker: sector} dictionary
stats["sector"] = stats.index.map(sector_of)

# Display sorted from best to worst Sharpe ratio, rounded to 3 decimals
stats.sort_values("sharpe", ascending=False).round(3)

### Risk and return together

The classic scatter. Each point is one stock; the eye is looking for whether
more risk actually bought more return over this particular five years.

**Plotting idioms.**

- `ax.scatter(x, y, s=70, edgecolor="white")` draws one marker per stock; `s`
  is the marker area, and a white edge separates overlapping points.
- `stats.iterrows()` loops over the rows of a DataFrame, giving
  `(index label, row as a Series)` pairs, so each point can be labelled with its
  ticker. (`iterrows` is slow on big tables; fine for 30 rows.)
- `ax.annotate(..., xytext=(0, 9), textcoords="offset points")` places the
  label 9 points *above* the data point rather than on top of it.
- `ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")` formats tick labels
  as percentages. A `lambda` is a one-line unnamed function; matplotlib calls
  it with the tick value and its position (ignored here as `_`).

**Look for:** no clean upward slope. High-volatility names such as CAT, GS
and BAC earned no more than lower-volatility UNH or HD. Over a single
five-year window, stock-specific stories swamp the risk-return trade-off that
theory predicts *on average*.

In [ ]:
# New 8 x 5.5 inch chart
fig, ax = plt.subplots(figsize=(8, 5.5))
# One dot per stock: volatility on x, return on y
ax.scatter(
    stats["ann_volatility"],  # x values
    stats["ann_return"],  # y values
    s=70,  # marker size (area in points squared)
    color="#2a78d6",  # blue fill
    edgecolor="white",  # white outline so overlapping dots stay distinguishable
    linewidth=1.5,  # thickness of that outline
    zorder=3,  # draw dots above the grid lines
)
# Loop over rows: ticker is the index label, row holds that stock's statistics
for ticker, row in stats.iterrows():
    # Label each dot with its ticker
    ax.annotate(
        ticker,  # the text
        (row["ann_volatility"], row["ann_return"]),  # the data point being labelled
        xytext=(0, 9),  # shift the text 0 right, 9 up ...
        textcoords="offset points",  # ... measured in points, not data units
        ha="center",  # horizontally centre the text over the dot
        fontsize=8,  # small text
        color="#52514e",  # dark grey
    )

# Dashed horizontal line at the risk-free rate: dots below it earned less than Treasuries
ax.axhline(rf_annual, color="#52514e", linewidth=0.8, linestyle="--")
# Label for that line, at the left edge of the chart
ax.annotate(
    f"risk-free proxy {rf_annual:.1%}",  # text, e.g. "risk-free proxy 2.3%"
    xy=(ax.get_xlim()[0], rf_annual),  # anchor: left end of the x-axis, at the risk-free level
    xytext=(4, 5),  # nudge 4 points right and 5 up
    textcoords="offset points",  # the nudge is in points
    fontsize=8,  # small text
    color="#52514e",  # dark grey
)
# Chart title
ax.set_title("Annualised risk and return, February 2013 – February 2018")
# x-axis label
ax.set_xlabel("Annualised volatility")
# y-axis label
ax.set_ylabel("Annualised return")
# Show x tick labels as whole percentages (0.2 -> "20%")
ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
# Same for the y-axis
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
# Turn on vertical grid lines only
ax.grid(axis="x")
# Render
plt.show()

### Exercise 1 (worked solution)

The chart above covers a single five-year bull market. Recompute `stats` for
calendar year 2015 only — a flat, choppy year — and compare the ranking. How
stable is a five-year Sharpe ratio as a guide to the next year?

**Approach.** We apply exactly the same formulas to the 252 trading days of
2015, using the average 10-year yield *for 2015* as the risk-free rate. The
result goes into a **new** table, `stats_2015`, so the five-year `stats`
(which later cells save to disk) is untouched.

- `simple_ret.loc["2015"]` uses pandas' *partial-string indexing*: on a date
  index, a year string selects every row in that year.
- `.rank(ascending=False)` gives 1 to the largest value, 2 to the next, and so
  on.
- **Spearman rank correlation** (`.corr(method="spearman")`) measures how
  similar two *orderings* are: +1 means identical ranking, 0 means no
  relationship, −1 means exactly reversed. It is the Pearson correlation of
  the ranks, so it ignores the size of the numbers and asks only "is the order
  preserved?"

Strictly, 2015 sits *inside* the five-year window, so the comparison flatters
the five-year number. A fairer "guide to the next year" test uses only data
known before 2015, so the solution also computes a Sharpe ratio on February
2013 to December 2014 and checks how well that ranking predicts 2015.

In [ ]:
# Exercise 1: recompute the risk-return table for calendar year 2015 only
# Partial-string indexing: every daily return dated in 2015 (rows), all 30 tickers (columns)
ret_2015 = simple_ret.loc["2015"]
# Risk-free proxy for 2015: average 10-year yield over the twelve 2015 months, as a decimal
rf_2015 = us10y.loc[us10y["date"].dt.year == 2015, "yield_pct"].mean() / 100

# Same formulas as the five-year table, but on 2015 data only; a NEW name so `stats` is untouched
stats_2015 = pd.DataFrame(
    {  # start of the dictionary
        "ann_return": ret_2015.mean() * TRADING_DAYS,  # annualised mean return in 2015
        "ann_volatility": ret_2015.std() * np.sqrt(TRADING_DAYS),  # annualised volatility in 2015
    }
)
# 2015 Sharpe ratio, using the 2015 risk-free proxy
stats_2015["sharpe"] = (stats_2015["ann_return"] - rf_2015) / stats_2015["ann_volatility"]

# Side-by-side table: five-year Sharpe vs 2015 Sharpe (pandas aligns the two Series on ticker)
sharpe_compare = pd.DataFrame({"sharpe_5y": stats["sharpe"], "sharpe_2015": stats_2015["sharpe"]})
# Rank 1 = highest five-year Sharpe; astype(int) turns 1.0 into 1 for tidier display
sharpe_compare["rank_5y"] = sharpe_compare["sharpe_5y"].rank(ascending=False).astype(int)
# Rank 1 = highest 2015 Sharpe
sharpe_compare["rank_2015"] = sharpe_compare["sharpe_2015"].rank(ascending=False).astype(int)
# Positive = the stock climbed the ranking in 2015; negative = it fell
sharpe_compare["places_moved_up"] = sharpe_compare["rank_5y"] - sharpe_compare["rank_2015"]
# Add sectors for context
sharpe_compare["sector"] = sharpe_compare.index.map(sector_of)

# Context line: how many trading days and what risk-free rate were used
print(f"2015: {len(ret_2015)} trading days, risk-free proxy {rf_2015:.2%}")
# How many of the 30 stocks beat the risk-free proxy in 2015
print(f"Stocks with a positive 2015 Sharpe ratio: {(stats_2015['sharpe'] > 0).sum()} of {len(stats_2015)}")
# Spearman correlation between the two rankings (1 = same order, 0 = unrelated)
rank_corr_in_sample = sharpe_compare["sharpe_5y"].corr(sharpe_compare["sharpe_2015"], method="spearman")
# Print it
print(f"Rank correlation, five-year Sharpe vs 2015 Sharpe:      {rank_corr_in_sample:.2f}")

# A fair forecasting test: Sharpe ratios using only data known BEFORE 2015 (Feb 2013 to Dec 2014)
ret_pre_2015 = simple_ret.loc[:"2014"]
# Risk-free proxy over the same pre-2015 months
rf_pre_2015 = us10y.loc[(us10y["date"] >= "2013-02-01") & (us10y["date"] <= "2014-12-31"), "yield_pct"].mean() / 100
# Pre-2015 Sharpe ratio for each stock, in one line: (annual return - rf) / annual volatility
sharpe_pre_2015 = (ret_pre_2015.mean() * TRADING_DAYS - rf_pre_2015) / (ret_pre_2015.std() * np.sqrt(TRADING_DAYS))
# How well the earlier ranking predicted the 2015 ranking
rank_corr_forecast = sharpe_pre_2015.corr(stats_2015["sharpe"], method="spearman")
# Print it
print(f"Rank correlation, 2013-14 Sharpe vs 2015 Sharpe:        {rank_corr_forecast:.2f}")

# Display the comparison, best 2015 Sharpe first
sharpe_compare.sort_values("rank_2015").round(3)

**Discussion.** The ranking is unstable, and that is the point of the exercise.

- 2015 really was a poor year: only 13 of the 30 stocks beat the 2.14%
  risk-free proxy, against 26 of 30 over the five years.
- The leaders changed. GOOGL (1.39), MCD (1.19) and NKE (1.18) top 2015.
  BA, the five-year number one at 1.42, falls to ninth (0.50). MMM drops 19
  places, from fifth to 24th (−0.49). GE, *dead last* over five years at
  −0.43, is fifth in 2015 at 0.95. WMT, 21st over five years, is last in 2015
  at −1.63.
- The Spearman rank correlation between the five-year and 2015 Sharpe ratios is
  only about 0.43 — and that is flattering, because 2015 is part of the
  five-year sample. Using only data available *before* 2015, the rank
  correlation with 2015 falls to about 0.28.

So a multi-year Sharpe ratio is a weak guide to next year's ranking. The
statistical reason is estimation error: the standard error of an annual
Sharpe ratio estimated from $T$ years is roughly $\sqrt{(1 + S^2/2)/T}$, about
0.5 for one year. A one-year Sharpe of 1.0 is statistically hard to tell from
zero. This is why fund selectors look at long track records, and why "past
performance is not a guide to future returns" is in every prospectus.

## 4. Rolling volatility, and realised vs implied

A single volatility number for five years hides everything interesting. Risk
is not constant: it clusters. Quiet periods follow quiet periods and violent
ones follow violent ones, which is the empirical fact that the whole GARCH
literature exists to model.

#### Background: rolling windows

`simple_ret.rolling(60).std()` slides a 60-row window down each column: on
each date it takes the standard deviation of that date and the 59 days before
it, then moves forward a day. The result has the same shape as the input; the
first 59 rows are `NaN` because a full window is not yet available. Multiply
by $\sqrt{252}$ to annualise as before.

The window length is a **bias-variance trade-off**. A short window (20 days)
reacts quickly to a new regime but is noisy; a long one (250 days) is smooth
but slow to notice that risk has changed. 60 days, roughly a quarter, is a
common compromise. A rolling window uses only *past and current* data, so it
is safe from look-ahead bias — which matters a lot in Notebook 03.

Industry models such as RiskMetrics replace the flat window with an
exponentially weighted one (`.ewm()` in pandas), so that recent days count
more; GARCH models go further and forecast tomorrow's variance from today's
shock and today's variance.

**Look for:** spikes that are shared across the three stocks — the August
2015 sell-off and the early-2016 sell-off — and long calm stretches in
between. That is volatility clustering. JNJ usually sits lowest (it averages
about 14% against about 23% for AAPL).

In [ ]:
# Window length in trading days (about three months)
WINDOW = 60  # roughly a quarter of trading days

# For each date and stock: std of the last 60 daily returns, annualised by sqrt(252)
rolling_vol = simple_ret.rolling(WINDOW).std() * np.sqrt(TRADING_DAYS)

# New wide, short chart
fig, ax = plt.subplots(figsize=(9.5, 4.5))
# Three stocks from different sectors: tech, bank, health care
for ticker in ["AAPL", "JPM", "JNJ"]:
    # One line per stock; the first 59 NaN values are simply not drawn
    ax.plot(rolling_vol.index, rolling_vol[ticker], label=ticker)
# Title built from the WINDOW constant, so it stays correct if WINDOW changes
ax.set_title(f"{WINDOW}-day rolling annualised volatility")
# y-axis label
ax.set_ylabel("Annualised volatility")
# No x-axis label
ax.set_xlabel("")
# Show y ticks as percentages
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
# Legend from the label= arguments
ax.legend()
# Render
plt.show()

### Realised vs implied volatility

The VIX is the options market's forecast of S&P 500 volatility over the next
30 days, quoted as an annualised percentage. Our equal-weight basket's
*realised* volatility is the same quantity measured after the fact, in the
same units — so the two belong on one axis.

(A chart with two different y-axes is almost always a mistake. If two series
are not in the same units, make two charts.)

**How the next cell builds the comparison.**

- `simple_ret.mean(axis=1)` averages **across columns** — across the 30
  stocks — on each date. (`axis=0`, the default, averages down rows.) The
  result is the daily return of an equal-weight portfolio, rebalanced daily.
  This works only because simple returns add across assets (section 1).
- The 21-day rolling standard deviation, times $\sqrt{252}$, times 100, puts
  realised volatility in the same units as the VIX: annualised percentage
  points over roughly one month (21 trading days $\approx$ 30 calendar days).
- `pd.DataFrame({...}).dropna()` aligns the two series on date and keeps only
  days present in both.
- `vol_compare.iloc[:, 1]` selects the second column by *position*
  (`iloc` = integer location; `:` means all rows).
- `(gap > 0).mean()`: a comparison gives True/False; the mean of booleans is
  the *fraction* of True values.

**Look for:** the orange VIX line above the blue realised line on about 91% of
days, with a median gap of about 4.4 volatility points.

In [ ]:
# Daily VIX closing values; set_index("date") makes the date the row label for alignment
vix = pd.read_csv(DATA / "vix_daily.csv", parse_dates=["date"]).set_index("date")

# Average across the 30 columns on each date (axis=1) = daily return of an equal-weight basket
basket_ret = simple_ret.mean(axis=1)  # equal-weight basket of our 30 names
# 21-day rolling volatility of the basket, annualised, then x100 to match the VIX's percentage units
realised_21d = basket_ret.rolling(21).std() * np.sqrt(TRADING_DAYS) * 100

# Two-column table aligned on date; dropna keeps only dates where both values exist
vol_compare = pd.DataFrame(
    {"realised (21-day, equal-weight basket)": realised_21d, "VIX (implied, S&P 500)": vix["vix_close"]}  # column name -> Series
).dropna()  # keep only dates where both series have a value

# New chart
fig, ax = plt.subplots(figsize=(9.5, 4.5))
# Second column (position 1) is the VIX, in orange
ax.plot(vol_compare.index, vol_compare.iloc[:, 1], label="VIX (implied)", color="#eb6834")
# First column (position 0) is realised volatility, in blue
ax.plot(vol_compare.index, vol_compare.iloc[:, 0], label="Realised (21-day)", color="#2a78d6")
# The title states the finding, not just the topic
ax.set_title("Implied volatility runs above realised volatility most of the time")
# y-axis label: both series share these units, so one axis is correct
ax.set_ylabel("Annualised volatility (%)")
# No x-axis label
ax.set_xlabel("")
# Legend
ax.legend()
# Render
plt.show()

# Daily gap: implied minus realised, in volatility points
gap = (vol_compare.iloc[:, 1] - vol_compare.iloc[:, 0])
# Fraction of days on which the gap was positive (mean of True/False = share of True)
print(f"VIX exceeded realised volatility on {(gap > 0).mean():.1%} of days.")
# The middle value of the gap, less sensitive to spikes than the mean
print(f"Median gap: {gap.median():.1f} volatility points.")

That persistent gap is the **variance risk premium** — buyers of options pay,
on average, more than the volatility that subsequently shows up. It is the
reason systematic option-selling strategies make money most of the time and
then occasionally lose several years of gains in a week.

(Note the apples-to-oranges caveat: VIX tracks the S&P 500, our basket is 30
large caps. The direction of the result is robust; the size of the gap is not
precise.)

## 5. Drawdown

Volatility is symmetric. Investors are not — they care about losses. Drawdown
is the peak-to-current decline, and maximum drawdown is the worst of them.
It is usually a better description of what a client actually experienced.

#### Background: drawdown and maximum drawdown

Let $W_t$ be the wealth path (growth of \$1). The running peak is
$M_t = \max_{s \le t} W_s$, the highest value reached so far. Then

$$DD_t = \frac{W_t}{M_t} - 1 \le 0, \qquad \text{MDD} = \min_t DD_t$$

$DD_t$ is zero at a new high and negative otherwise; maximum drawdown (MDD) is
the deepest point, the worst peak-to-trough loss an investor who bought at the
top would have suffered.

Why practitioners like it: it is path-dependent (volatility is not — shuffle
the daily returns and volatility is unchanged, but drawdown changes), it
describes a real experience ("you were down 55%"), and it is what makes
clients redeem and risk managers cut positions. Hedge-fund mandates often
carry hard drawdown limits. Weaknesses: it is a single worst event, so it is
noisy and grows with the length of the sample, and it says nothing about how
long recovery took.

**How the code does it.**

- Inside `drawdown()`: `cumprod()` builds the wealth path, `cummax()` is the
  running maximum ("highest so far") down the column, and the last line is the
  formula above.
- `close.apply(func)` calls `func` once **per column**, passing that column as
  a Series, and stacks the results back into a DataFrame. The `lambda col:`
  converts each price column to returns, then to drawdowns.
- `dd.min()` takes the most negative value in each column — the maximum
  drawdown — and `sort_values()` puts the worst first.

**Look for:** GE at about −54.7%, followed by CVX and CAT around −48% (the
2014–2016 oil and commodity slump).

In [ ]:
# Define a reusable drawdown function that works on any return series
def drawdown(returns: pd.Series) -> pd.Series:
    # The docstring below is the function's built-in help text
    """Peak-to-current decline of a cumulative return series."""
    # Wealth path of $1: cumulative product of (1 + return); the first NaN becomes 0%
    wealth = (1 + returns.fillna(0)).cumprod()
    # Highest wealth reached up to and including each date
    running_peak = wealth.cummax()
    # Current wealth relative to that peak, minus 1: 0 at a new high, negative below it
    return wealth / running_peak - 1


# Apply column by column: turn each stock's prices into returns, then into a drawdown path
dd = close.apply(lambda col: drawdown(col.pct_change()))

# Deepest drawdown per stock (most negative value), sorted worst first
max_dd = dd.min().sort_values()
# Heading
print("Worst maximum drawdowns:")
# Six worst, converted to percent and rounded to 1 decimal; to_string prints a plain list
print((max_dd.head(6) * 100).round(1).to_string())

### The worst drawdown, drawn

The next cell plots the drawdown path of whichever stock had the worst maximum
drawdown. `max_dd.index[0]` is the first ticker in the sorted Series, so the
chart picks the stock automatically rather than hard-coding "GE".

`ax.fill_between(x, y1, y2, alpha=0.25)` shades the area between two curves —
here between the drawdown line and zero — with 25% opacity. A shaded
"underwater" chart is the standard way risk reports show drawdown.

**Look for:** the shape of GE's slide. Its maximum drawdown is measured from
a peak in July 2016, and the line falls almost without a break to its lowest
point in February 2018, the very end of the sample, so a holder who bought at
that peak had lost more than half the money and had seen no recovery at all.
Time under water is as important to a client as depth.

In [ ]:
# New wide chart
fig, ax = plt.subplots(figsize=(9.5, 4))
# The ticker with the worst maximum drawdown (first entry after sorting)
worst = max_dd.index[0]
# Shade the area between the drawdown line (in percent) and zero, light blue
ax.fill_between(dd.index, dd[worst] * 100, 0, color="#2a78d6", alpha=0.25)
# Draw the drawdown line itself on top of the shading
ax.plot(dd.index, dd[worst] * 100, color="#2a78d6")
# Title includes the ticker and its maximum drawdown, formatted as a percentage
ax.set_title(f"{worst} drawdown — worst in the sample at {max_dd.iloc[0]:.1%}")
# y-axis label
ax.set_ylabel("Drawdown (%)")
# No x-axis label
ax.set_xlabel("")
# Render
plt.show()

### Exercise 2 (worked solution)

Add a column to `stats` called `max_drawdown`. Then compute the **Calmar
ratio** — annualised return divided by the absolute value of maximum drawdown
— and rank the stocks by it. Which names move up the ranking compared with the
Sharpe ordering, and why?

**Background: the Calmar ratio.**

$$\text{Calmar} = \frac{\bar R_{\text{annual}}}{|\text{MDD}|}$$

Same numerator idea as Sharpe (here without subtracting the risk-free rate,
following the usual definition), but the risk measure in the denominator is
the worst peak-to-trough loss rather than volatility. It answers "how many
years of average return would it take to earn back the worst loss?" It is
popular with commodity-trading advisers and hedge funds, which are judged by
clients who remember drawdowns.

**Approach.** The later "Save the outputs" cell writes `stats` to a CSV, so to
keep that file identical to the original we add the column to a **copy**,
`stats_calmar`, rather than to `stats` itself. `max_dd` from section 5 is a
Series indexed by ticker, so assigning it as a column lines up by ticker
automatically (index alignment), whatever order the rows are in.

In [ ]:
# Exercise 2: Calmar ratio. Work on a copy so the `stats` table saved later is unchanged
stats_calmar = stats.copy()
# Maximum drawdown per stock from section 5; pandas matches rows by ticker (index alignment)
stats_calmar["max_drawdown"] = max_dd
# Calmar ratio = annualised return / |maximum drawdown|
stats_calmar["calmar"] = stats_calmar["ann_return"] / stats_calmar["max_drawdown"].abs()

# Rank 1 = highest Sharpe ratio
stats_calmar["sharpe_rank"] = stats_calmar["sharpe"].rank(ascending=False).astype(int)
# Rank 1 = highest Calmar ratio
stats_calmar["calmar_rank"] = stats_calmar["calmar"].rank(ascending=False).astype(int)
# Positive = the stock ranks better on Calmar than on Sharpe
stats_calmar["places_moved_up"] = stats_calmar["sharpe_rank"] - stats_calmar["calmar_rank"]

# How similar the two orderings are overall (Spearman rank correlation)
print(f"Rank correlation, Sharpe vs Calmar: {stats_calmar['sharpe'].corr(stats_calmar['calmar'], method='spearman'):.2f}")
# Columns to show, in a readable order
calmar_cols = ["ann_return", "ann_volatility", "max_drawdown", "sharpe", "calmar",
               "sharpe_rank", "calmar_rank", "places_moved_up", "sector"]  # continuation of the column list
# Display best Calmar first, rounded to 3 decimals
stats_calmar.sort_values("calmar", ascending=False)[calmar_cols].round(3)

**Discussion.** At the top and bottom the two measures agree — the rank
correlation is about 0.96 and the bottom six (DUK, PG, T, CVX, XOM, GE) do
not move at all — but the middle reshuffles, and the pattern is
informative.

- **Moved up:** MCD (+6 places, 15th to 9th; max drawdown only −16.0%), AMT
  (+5, drawdown −20.3%), KO (+3, −15.3%), UNH (2nd to 1st; drawdown just −13.2%,
  the shallowest in the sample), and NEE, JNJ, MSFT, GOOGL, CSCO and PFE by one
  or two places each.
- **Moved down:** BAC (−6, 9th to 15th; drawdown −39.5%), BA (−4, 1st to 5th;
  drawdown −31.5%), INTC (−4, −31.3%), DIS (−3, −27.0%), and MMM, AAPL, CAT
  and WMT (−2 each).

**Why.** Volatility measures the size of *all* day-to-day wiggles, up and
down. Maximum drawdown measures one sustained slide. Steady defensive or
low-beta names (MCD, KO, UNH, JNJ, NEE) can have ordinary volatility but
shallow drawdowns, because their bad days are not clustered into a long
decline. Banks, cyclicals and chipmakers (BAC, BA, INTC, CAT) each suffered one
long, deep slide — BAC from July 2015 and BA from February 2015, both to a
low in February 2016; INTC from December 2014 to August 2015; CAT from mid-2014
to January 2016 in the commodity slump — which Calmar punishes far more than
Sharpe does. Remember that Calmar rests on a *single* worst episode, so it is
even noisier than Sharpe; the two together tell you more than either alone.

## 6. Correlation and diversification

Diversification works when assets do not move together. The correlation
matrix is how you check, and it is the input to every portfolio optimiser.

#### Background: correlation and diversification

**Pearson correlation** between two return series is their covariance scaled
by both volatilities:

$$\rho_{ij} = \frac{\operatorname{Cov}(R_i, R_j)}{\sigma_i\,\sigma_j}, \qquad -1 \le \rho_{ij} \le 1$$

It measures how *linearly* the two move together: +1 lockstep, 0 unrelated, −1
mirror images. `simple_ret.corr()` computes it for every pair of columns and
returns a 30 × 30 matrix: symmetric, with 1s on the diagonal (each stock with
itself). Missing values are handled pair by pair.

**Why it drives diversification.** For an equal-weight portfolio of $N$ stocks
that each have volatility $\sigma$ and pairwise correlation $\rho$:

$$\sigma_p^2 = \frac{\sigma^2}{N} + \frac{N-1}{N}\,\rho\,\sigma^2 \;\xrightarrow{\;N \to \infty\;}\; \rho\,\sigma^2$$

The first term (stock-specific risk) diversifies away as $N$ grows. The second
(shared, market risk) does not. With $\rho = 0.32$, roughly the average
cross-sector correlation found below, even an infinitely diversified portfolio
keeps $\sqrt{0.32} \approx 57\%$ of a single stock's volatility. Correlation sets
the floor on how much risk diversification can remove. This is also why
covariance matrices are the input to Markowitz mean-variance optimisation.

**The next cell's code.**

- `sorted(corr.columns, key=lambda t: (sector, ticker))` orders tickers by
  sector, then alphabetically. Sorting by a tuple sorts by its first element,
  breaking ties with the second. `sector_of.get(t, "zzz")` returns `"zzz"` for
  any ticker missing from the dictionary, pushing it to the end.
- `corr.loc[order, order]` reorders both rows and columns.
- `np.eye(n, dtype=bool)` is an $n \times n$ True/False matrix with True on the
  diagonal; `~` flips it, and `corr.where(mask)` keeps values where the mask is
  True and puts `NaN` elsewhere — so the trivial 1.0 diagonal is excluded.
  `.min().min()` takes the column minimums, then the minimum of those.

**Look for:** correlations from about −0.04 to 0.83. Almost nothing is
negative.

In [ ]:
# 30 x 30 matrix of pairwise Pearson correlations between the stocks' daily returns
corr = simple_ret.corr()

# Order the tickers by sector so the block structure is visible
# Sort key is a (sector, ticker) tuple; "zzz" sends any ticker with no sector to the end
order = sorted(corr.columns, key=lambda t: (sector_of.get(t, "zzz"), t))
# Reorder rows and columns into that sector-grouped order
corr = corr.loc[order, order]

# Blank out the diagonal (each stock with itself, always 1.0): ~np.eye(...) is True off the diagonal
off_diagonal = corr.where(~np.eye(len(corr), dtype=bool))
# Smallest and largest off-diagonal correlations: min of each column, then min of those
print(f"Correlations run from {off_diagonal.min().min():.2f} to {off_diagonal.max().max():.2f}")

### Choosing the colour scale — a two-minute detour that matters

Correlation ranges from −1 to +1 with a meaningful zero, so the textbook
choice is a **diverging** scale: one hue for negative, another for positive,
a neutral grey at zero.

But we just checked, and *every* pair in this sample is essentially
non-negative. Spending half a diverging scale on values that do not occur
leaves every real value washed out. So we use a **sequential** single-hue
scale from 0 to 1, which puts the whole ramp to work.

The general rule: diverging when the data actually straddles a meaningful
midpoint, sequential when it does not. Check first, then choose. (Both
colour maps are built below so you can switch and see the difference.)

**How the heatmap is drawn.**

- `LinearSegmentedColormap.from_list(name, [colours])` builds a smooth
  colour scale through the listed colours: white → light blue → blue → navy
  for `sequential`; blue → light grey → red for `diverging`.
- `SECTOR_SHORT` is a dictionary of short sector names for the axis labels.
- `ax.imshow(matrix, cmap=..., vmin=0, vmax=1)` draws a 2-D array as a grid of
  coloured cells. `vmin`/`vmax` pin the ends of the colour scale, so 0 is
  always white and 1 always navy regardless of the data.
- `ax.set_xticks(positions, labels, rotation=90)` places and labels the ticks;
  the y labels are built with a list comprehension that joins ticker and short
  sector name.
- `fig.colorbar(im, ax=ax)` adds the legend bar that maps colours to values.

(The notebook builds its heatmap directly in matplotlib. The popular
**seaborn** library wraps the same thing in one call, `sns.heatmap`, but is
not needed here.)

**To try the diverging scale**, change `cmap=sequential, vmin=0` to
`cmap=diverging, vmin=-1` and re-run: most cells turn a washed-out pink.

**Look for:** darker blocks along the diagonal — Financials (BAC, GS, JPM,
WFC) and Energy (CVX, XOM) especially — and pale rows for the Utilities
(DUK, NEE).

In [ ]:
# Single-hue sequential scale: white (0) -> light blue -> blue -> navy (1)
sequential = LinearSegmentedColormap.from_list(
    "fin4600_sequential", ["#ffffff", "#9ec5f4", "#2a78d6", "#0d366b"]  # name, then the colour stops
)
# Two-hue diverging scale: blue (negative) -> light grey (zero) -> red (positive); built for comparison
diverging = LinearSegmentedColormap.from_list(
    "fin4600_diverging", ["#2a78d6", "#f0efec", "#e34948"]  # name, then the colour stops
)

# Short sector names so the y-axis labels fit
SECTOR_SHORT = {
    "Communication Services": "Comm Svcs",  # full GICS name -> abbreviation
    "Consumer Discretionary": "Cons Disc",  # full GICS name -> abbreviation
    "Consumer Staples": "Cons Stpl",  # full GICS name -> abbreviation
    "Energy": "Energy",  # already short
    "Financials": "Financials",  # already short
    "Health Care": "Health",  # full GICS name -> abbreviation
    "Industrials": "Industrials",  # already short
    "Information Technology": "Info Tech",  # full GICS name -> abbreviation
    "Materials": "Materials",  # already short
    "Real Estate": "Real Est",  # full GICS name -> abbreviation
    "Utilities": "Utilities",  # already short
}

# Near-square figure for a square matrix
fig, ax = plt.subplots(figsize=(9.5, 8))
# Draw the matrix as coloured cells; vmin/vmax fix the colour scale to the 0-to-1 range
im = ax.imshow(corr.values, cmap=sequential, vmin=0, vmax=1)
# One x tick per ticker, labelled and rotated 90 degrees so the labels do not overlap
ax.set_xticks(range(len(order)), order, rotation=90, fontsize=8)
# One y tick per ticker, labelled "TICKER  ·  Sector"
ax.set_yticks(
    range(len(order)),  # tick positions 0..29
    [f"{t}  ·  {SECTOR_SHORT.get(sector_of.get(t), '?')}" for t in order],  # label for each ticker; '?' if unknown
    fontsize=8,  # small labels
)
# Title
ax.set_title("Daily return correlation, tickers grouped by GICS sector")
# Grid lines would cut across the cells, so turn them off
ax.grid(False)
# Colour legend bar beside the chart; fraction and pad control its width and spacing
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
# Label on the colour bar
cbar.set_label("Correlation", fontsize=9, color="#52514e")
# Render
plt.show()

### Within-sector vs across-sector correlation

The heatmap suggests same-sector pairs are more correlated. The next cell
measures it. It loops over every **unique pair** of tickers (435 of them:
$30 \times 29 / 2$) — `enumerate(order)` gives each ticker with its position
`i`, and `order[i + 1 :]` gives only the tickers after it, so each pair
appears once and no ticker is paired with itself. Each pair becomes a
dictionary in a list, and the list becomes a DataFrame.

Then:

- `groupby("same_sector")["corr"].agg(["count", "mean", "median"])` splits
  the pairs into same-sector (True) and cross-sector (False) and summarises
  each group.
- `nlargest(5, "corr")` / `nsmallest(5, "corr")` return the five rows with the
  highest / lowest correlation.

**Look for:** 35 same-sector pairs averaging about 0.49 against 400
cross-sector pairs averaging about 0.32. The top five pairs are all banks
except CVX–XOM; the bottom five all pair a bank with a utility.

In [ ]:
# Average correlation within a sector vs across sectors
# Empty list to collect one dictionary per pair of stocks
pairs = []
# enumerate gives each ticker together with its position i in the sector-sorted list
for i, a in enumerate(order):
    # Only tickers AFTER position i, so each pair is counted once and never a stock with itself
    for b in order[i + 1 :]:
        # Record this pair as a dictionary
        pairs.append(
            {  # start of the dictionary
                "a": a,  # first ticker
                "b": b,  # second ticker
                "corr": corr.loc[a, b],  # their correlation, read from the matrix
                "same_sector": sector_of.get(a) == sector_of.get(b),  # True if both are in the same sector
            }
        )
# Turn the list of dictionaries into a table: one row per pair, 435 rows
pair_df = pd.DataFrame(pairs)

# Count, mean and median correlation for cross-sector (False) and same-sector (True) pairs
print(pair_df.groupby("same_sector")["corr"].agg(["count", "mean", "median"]).round(3))
# Heading
print("\nMost correlated pairs:")
# Five highest correlations; index=False hides the row numbers
print(pair_df.nlargest(5, "corr")[["a", "b", "corr"]].to_string(index=False))
# Heading
print("\nLeast correlated pairs:")
# Five lowest correlations
print(pair_df.nsmallest(5, "corr")[["a", "b", "corr"]].to_string(index=False))

Same-sector pairs are more correlated than cross-sector pairs — which is
exactly why sector limits exist in portfolio mandates.

The uncomfortable part: correlations are not stable. They rise in crises,
precisely when diversification is supposed to help. Check it.

The next cell computes a **rolling correlation**: `A.rolling(120).corr(B)`
gives, on each date, the correlation between the two stocks over the previous
120 trading days (about six months). It is the same rolling-window idea as
rolling volatility, applied to a pair. `ax.set_ylim(0, 1)` fixes the y-axis so
the size of the moves is not exaggerated.

**Look for:** even for two very similar banks the correlation swings between
about 0.61 (late 2014, a calm period) and 0.94 (January 2016, in the middle of
the early-2016 sell-off), and it stays near 0.9 from late 2015 onwards. Any
risk model that uses one fixed correlation is wrong for most of the sample.

In [ ]:
# 120-day rolling correlation between JPM's and BAC's daily returns
rolling_corr = simple_ret["JPM"].rolling(120).corr(simple_ret["BAC"])

# New chart
fig, ax = plt.subplots(figsize=(9.5, 4))
# Plot the rolling correlation over time
ax.plot(rolling_corr.index, rolling_corr, color="#2a78d6")
# Title
ax.set_title("120-day rolling correlation: JPM and BAC")
# y-axis label
ax.set_ylabel("Correlation")
# No x-axis label
ax.set_xlabel("")
# Fix the y-axis to 0..1 so swings are shown at honest scale
ax.set_ylim(0, 1)
# Render
plt.show()

# Lowest and highest rolling correlation (NaN values in the first 119 rows are ignored)
print(f"Range over the sample: {rolling_corr.min():.2f} to {rolling_corr.max():.2f}")

## 7. The distribution of returns, and Value at Risk

Much of finance theory assumes returns are normally distributed. They are
not. They have **fat tails**: extreme moves happen far more often than a
normal distribution predicts. Every risk model that ignored this has
eventually embarrassed someone.

#### Background: SciPy

**SciPy** ("scientific Python") is the library of scientific algorithms built
on NumPy: optimisation, integration, linear algebra, signal processing and,
in `scipy.stats`, probability distributions and statistical tests. In finance
it is the standard toolkit for distribution work: normal and t
densities, quantiles for VaR, option-pricing integrals, curve fitting and
optimisation of portfolios. We import it as `scipy_stats` because the name
`stats` is already taken by our risk-return table — reusing it would silently
overwrite the table.

A distribution object such as `scipy_stats.norm` has a consistent set of
methods:

| Method | Meaning | Example |
|---|---|---|
| `pdf(x, loc, scale)` | probability density at `x` (height of the bell curve) | used for the overlay below |
| `cdf(x)` | probability of a value $\le x$ | `norm.cdf(-1.645)` $\approx 0.05$ |
| `sf(x)` | survival function, $1 - \text{cdf}(x)$, probability of a value $> x$ | `norm.sf(3)` $\approx 0.00135$ |
| `ppf(q)` | inverse cdf: the value with probability `q` below it (a quantile) | `norm.ppf(0.01)` $\approx -2.326$ |

`loc` is the mean and `scale` the standard deviation; with neither, it is the
standard normal (mean 0, standard deviation 1).

#### Background: comparing a histogram with a normal curve

A **histogram** (`ax.hist(data, bins=80)`) splits the range of the data into
80 equal-width bins and draws a bar for the number of days in each. The
normal density integrates to 1, so to sit on the same scale as counts it must
be multiplied by (number of observations × bin width). The bin width here is
$(\max - \min)/80$ in percentage points, which is exactly the scaling factor in
the code. `np.linspace(a, b, 400)` creates 400 evenly spaced x values to draw
a smooth curve.

**Look for:** a histogram that is *taller and narrower* in the middle than the
orange normal curve, with a scattering of bars far out in both tails where the
normal curve is essentially zero.

In [ ]:
# SciPy's statistics module; aliased so it does not clash with our `stats` DataFrame
from scipy import stats as scipy_stats

# The stock to study in this section
ticker = "JPM"
# JPM's daily simple returns, with the initial NaN removed
r = simple_ret[ticker].dropna()

# New chart
fig, ax = plt.subplots(figsize=(9, 4.5))
# Histogram of daily returns in percent: 80 equal-width bins, bar height = number of days
ax.hist(r * 100, bins=80, color="#2a78d6", alpha=0.85)

# Overlay the normal distribution with the same mean and standard deviation
# 400 evenly spaced x values spanning the observed range, in percent
x = np.linspace(r.min() * 100, r.max() * 100, 400)
# Normal density at each x, with JPM's own mean and standard deviation (in percent)
normal_pdf = scipy_stats.norm.pdf(x, r.mean() * 100, r.std() * 100)
# Rescale density to counts: x number of days x bin width ((max - min) in % / 80 bins)
ax.plot(x, normal_pdf * len(r) * (r.max() - r.min()) * 100 / 80,
        color="#eb6834", linewidth=2, label="Normal distribution, same mean and SD")  # styling for the curve

# Title uses the ticker variable
ax.set_title(f"{ticker} daily returns vs a normal distribution")
# x-axis label
ax.set_xlabel("Daily return (%)")
# y-axis label: bar heights are day counts
ax.set_ylabel("Number of days")
# Legend
ax.legend()
# Render
plt.show()

#### Background: skewness, excess kurtosis and fat tails

The mean and standard deviation describe the centre and width of a
distribution. The next two *moments* describe its shape. With
$z_t = (R_t - \bar R)/s$ the standardised return:

- **Skewness** $\approx \frac{1}{n}\sum z_t^3$ measures asymmetry. Cubing keeps
  the sign, so a long left tail (rare big losses) gives negative skew, a long
  right tail positive skew. A normal distribution has skew 0. Equity *indices*
  are usually negatively skewed (crashes are sharper than rallies); single
  stocks less so.
- **Kurtosis** $\approx \frac{1}{n}\sum z_t^4$ measures tail weight. Raising to
  the fourth power makes the rare large $z$ values dominate. A normal
  distribution has kurtosis 3, so we usually report **excess kurtosis** =
  kurtosis − 3, which is 0 for a normal. Positive excess kurtosis
  (*leptokurtic*, "fat-tailed") means more mass in the centre *and* in the
  tails, less in the shoulders — exactly the tall, narrow histogram above.

pandas' `r.skew()` and `r.kurtosis()` compute bias-corrected sample versions of
these, and **`kurtosis()` already returns excess kurtosis** (the Fisher
definition), so 0 is the normal benchmark.

**A direct tail count.** Under a normal distribution, the probability of a move
beyond 3 standard deviations in *either* direction is
$2 \times P(Z > 3) = 2 \times$ `norm.sf(3)` $\approx 0.27\%$, so over $n$ days we
expect $0.0027\,n$ such days. The cell compares that with the number actually
observed.

**Look for:** skewness about 0.04 (essentially symmetric), excess kurtosis about
3.39, and 17 days beyond three standard deviations where a normal
distribution predicts about 3.4 — five times as many. That is what "fat
tails" means in practice.

In [ ]:
# Header line
print(f"{ticker} daily returns")
# Average daily return, shown as a percentage to 4 decimals
print(f"  Mean:               {r.mean():.4%}")
# Daily sample standard deviation (daily volatility)
print(f"  Standard deviation: {r.std():.4%}")
# Sample skewness: asymmetry of the distribution
print(f"  Skewness:           {r.skew():.3f}   (0 for a normal distribution)")
# pandas kurtosis() returns EXCESS kurtosis (normal = 0), so it can be read directly
print(f"  Excess kurtosis:    {r.kurtosis():.3f}   (0 for a normal distribution)")

# How often did moves beyond 3 standard deviations actually happen?
# Size of a three-standard-deviation daily move
threshold = 3 * r.std()
# Count days whose absolute return exceeded it (True counts as 1 when summed)
actual = (r.abs() > threshold).sum()
# Normal prediction: P(|Z| > 3) = 2 x sf(3), times the number of days
expected = scipy_stats.norm.sf(3) * 2 * len(r)
# Heading (an f-string with nothing to fill in, so it prints as written)
print(f"\nDays beyond 3 standard deviations")
# The expected count under normality
print(f"  Predicted by the normal distribution: {expected:.1f}")
# The observed count
print(f"  Actually observed:                    {actual}")
# How many times more often than the normal model predicts
print(f"  Ratio:                                {actual / expected:.1f}x")

### Historical Value at Risk

**VaR at 99% over one day** is the loss that is exceeded on only 1% of days.
It is a quantile of the return distribution, and there are two common ways to
get it:

- **Parametric**: assume normality and use $\mu - 2.326\sigma$. Fast, and
  wrong in the direction that matters.
- **Historical**: take the empirical 1st percentile. Makes no distributional
  assumption, but can only tell you about losses that have already happened.

**Expected Shortfall** (average loss *given* that VaR was breached) is the
measure Basel III moved to, precisely because VaR says nothing about how bad
the tail is.

#### Background: Value at Risk, two ways, and Expected Shortfall

**Definition.** For confidence level $c$ (say 99%), one-day VaR is the loss
$L$ such that $P(\text{loss} > L) = 1 - c$. Equivalently, it is minus the
$(1-c)$ quantile of the return distribution, times the position size:

$$\text{VaR}_c = -\,q_{1-c}(R) \times \text{position}$$

**Historical VaR** uses the empirical quantile. `r.quantile(0.01)` sorts the
1,258 daily returns and reads off the value with 1% of observations below it
(interpolating between neighbouring observations). No distribution is
assumed, so fat tails are captured automatically — but only the tails that
happened to occur in the sample window.

**Parametric (variance-covariance) VaR** assumes returns are normal with the
sample mean $\mu$ and standard deviation $\sigma$:

$$q_{1-c} = \mu + z_{1-c}\,\sigma, \qquad z_{0.05} = -1.645, \quad z_{0.01} = -2.326$$

where $z_{1-c}$ is `scipy_stats.norm.ppf(1 - c)`, the standard normal quantile.
It is fast, scales to large portfolios through the covariance matrix, and was
the original JP Morgan RiskMetrics approach. Its flaw is the normal
assumption: with positive excess kurtosis the true 1% quantile lies further
out than $\mu - 2.326\sigma$.

**Expected Shortfall** (ES, also called CVaR) is the average return on the days
that were at or beyond the VaR quantile:

$$\text{ES}_c = -\,E[R \mid R \le q_{1-c}]$$

In code, `r[r <= historical_var].mean()`: a boolean filter keeps the tail days
and `.mean()` averages them. VaR answers "how bad is a bad day?"; ES answers
"when it is bad, how bad on average?". Basel III's market-risk rules (FRTB)
replaced 99% VaR with 97.5% ES for this reason; ES is also *sub-additive*
(diversifying never increases it), which VaR does not guarantee.

**Look for:** at 95% the two VaR methods agree almost exactly (about \$20,400
on \$1m). At 99%, historical VaR is about \$35,200 but parametric only about
\$29,100: the normal model understates the 1-in-100-day loss by about a sixth.
Expected Shortfall at 99% is larger still, about \$43,300.

Note the f-string formats: `{position:,}` adds thousands separators and
`{value:>10,.0f}` right-aligns in a 10-character field with no decimals, so
the dollar amounts line up in a column.

In [ ]:
# Size of the hypothetical holding in dollars (underscores are just digit separators)
position = 1_000_000  # a $1m position

# Repeat the calculation at the two standard confidence levels
for confidence in (0.95, 0.99):
    # Historical VaR: the empirical 5% or 1% quantile of daily returns (a negative number)
    historical_var = r.quantile(1 - confidence)
    # Parametric VaR: mean + z x std, with z = norm.ppf(0.05) = -1.645 or norm.ppf(0.01) = -2.326
    parametric_var = r.mean() + scipy_stats.norm.ppf(1 - confidence) * r.std()
    # Expected Shortfall: average of all returns at or below the historical VaR quantile
    expected_shortfall = r[r <= historical_var].mean()

    # Heading for this confidence level; {position:,} prints 1,000,000
    print(f"{confidence:.0%} confidence, one-day horizon, ${position:,} position in {ticker}")
    # Minus sign turns the negative return quantile into a positive loss; times position gives dollars
    print(f"  Historical VaR:     ${-historical_var * position:>10,.0f}   ({-historical_var:.2%})")
    # Same for the normal-distribution estimate
    print(f"  Parametric VaR:     ${-parametric_var * position:>10,.0f}   ({-parametric_var:.2%})")
    # And for Expected Shortfall
    print(f"  Expected Shortfall: ${-expected_shortfall * position:>10,.0f}   ({-expected_shortfall:.2%})")
    # Blank line between the two confidence levels
    print()

Notice that parametric VaR is the *smaller* number at 99%. Assuming normality
makes the risk look better than it is — which is the one direction you never
want a risk model to be wrong in.

## 8. Beta, two ways

Beta is the sensitivity of a stock to the market. We do not have the S&P 500
index in this repo, so we build an equal-weight basket of our 30 names as a
market proxy. (A real analysis would use a cap-weighted index; the basket is
a teaching stand-in.)

$$\beta_i = \frac{\operatorname{Cov}(R_i, R_m)}{\operatorname{Var}(R_m)}$$

...which is also the slope coefficient of a regression of the stock's returns
on the market's. Computing it both ways is a good habit: if two independent
routes to a number disagree, one of them has a bug.

#### Background: beta by regression

**The model.** The market model (the empirical form of the CAPM) regresses a
stock's return on the market's:

$$R_{i,t} = \alpha_i + \beta_i R_{m,t} + \varepsilon_{i,t}$$

$\beta_i$ is the stock's sensitivity to the market (a beta of 1.4 means the
stock tends to move 1.4% for each 1% market move), $\alpha_i$ is the intercept
(average return not explained by the market), and $\varepsilon$ is
stock-specific noise, which diversifies away. In the CAPM, beta — not total
volatility — is the risk that earns a premium, because it is the part that
cannot be diversified (section 6).

**Ordinary least squares (OLS)** picks $\alpha$ and $\beta$ to minimise the sum
of squared residuals $\sum_t \varepsilon_t^2$. Writing the regressors as a
*design matrix* $X$ — a column of 1s (for the intercept) next to the market
return — and the stock's returns as $y$, the solution is

$$\hat b = \begin{pmatrix}\hat\alpha \\ \hat\beta\end{pmatrix} = (X^\top X)^{-1} X^\top y$$

and for a single regressor plus intercept this simplifies to exactly
$\hat\beta = \operatorname{Cov}(R_i, R_m)/\operatorname{Var}(R_m)$. So the two
routes must agree to floating-point precision.

**The code, route by route.**

- `market = simple_ret.mean(axis=1)`: the equal-weight basket again, as the
  market proxy.
- **Route 1:** `simple_ret.apply(lambda col: col.cov(market) / market.var())`
  runs the covariance formula on each stock's column.
- **Route 2:** `simple_ret.dropna()` keeps only days with all 30 returns, so
  $X$ and $y$ have the same rows. `np.column_stack([ones, market])` glues the
  column of 1s and the market column into the $n \times 2$ design matrix.
  `np.linalg.lstsq(X, y, rcond=None)` is NumPy's least-squares solver; it
  returns a tuple (coefficients, residuals, rank, singular values), and
  `coefficients, *_ = ...` keeps the first item and discards the rest.
  `coefficients[1]` is $\hat\beta$ (`[0]` would be $\hat\alpha$).

**Caveats.** Each stock is 1/30 of the basket, so it partly regresses on
itself, which pulls betas slightly towards 1; a real analysis would use the
S&P 500 and usually excess returns over the risk-free rate. Beta estimates
are also noisy and drift over time, which is why data vendors often "shrink"
raw betas towards 1 (Blume adjustment).

**Look for:** the largest disagreement between methods is about $10^{-15}$ —
floating-point rounding, i.e. identical. Banks have the highest betas (BAC
about 1.48); utilities the lowest (DUK about 0.53).

In [ ]:
# Market proxy: equal-weight average return of the 30 stocks each day, named "market"
market = simple_ret.mean(axis=1).rename("market")

# Route 1: covariance over variance
# For each stock column: Cov(stock, market) / Var(market); returns a Series indexed by ticker
cov_betas = simple_ret.apply(lambda col: col.cov(market) / market.var())

# Route 2: least-squares regression slope
# Keep only dates where every stock has a return (drops the first NaN row)
aligned = simple_ret.dropna()
# Market returns on exactly those same dates
market_aligned = market.loc[aligned.index]
# Design matrix X: a column of 1s (intercept, alpha) next to the market return (slope, beta)
design = np.column_stack([np.ones(len(market_aligned)), market_aligned.values])
# Empty dictionary to collect {ticker: regression beta}
ols_betas = {}
# Run one regression per stock
for col in aligned.columns:
    # Least-squares solve of X b = y; keep the coefficient vector [alpha, beta], discard the rest
    coefficients, *_ = np.linalg.lstsq(design, aligned[col].values, rcond=None)
    # Store the slope, which is beta
    ols_betas[col] = coefficients[1]

# Put both estimates in one table, aligned by ticker
beta_table = pd.DataFrame(
    {"beta_covariance": cov_betas, "beta_regression": pd.Series(ols_betas)}  # dict -> Series so pandas can align
)
# Difference between the two routes: should be zero up to rounding
beta_table["difference"] = beta_table["beta_covariance"] - beta_table["beta_regression"]
# Sector label for each ticker
beta_table["sector"] = beta_table.index.map(sector_of)

# Largest absolute disagreement, in scientific notation (e.g. 1.55e-15)
print(f"Largest disagreement between the two methods: {beta_table['difference'].abs().max():.2e}")
# Display highest beta first
beta_table.sort_values("beta_covariance", ascending=False).round(3)

### Betas as a bar chart

The next cell draws a horizontal bar chart of all 30 betas, sorted from
lowest to highest.

- The list comprehension
  `["#eb6834" if b > 1 else "#2a78d6" for b in plot_betas.values]` builds one
  colour per bar: orange for betas above 1 (more sensitive than the market),
  blue for below.
- `ax.barh(labels, values)` draws horizontal bars, which suit long lists of
  category labels better than vertical ones.
- `ax.axvline(1.0)` marks the market beta of 1.
- The final loop uses `enumerate` to get each bar's position `y` (0, 1, 2, ...)
  and writes the value just past the end of the bar with `ax.text`.

**Look for:** the four banks and the tech names in orange at the top, the
Utilities, Consumer Staples and telecoms (VZ, T) in blue at the bottom.

In [ ]:
# Covariance betas sorted ascending, so the largest bar ends up at the top of the chart
plot_betas = beta_table["beta_covariance"].sort_values()

# Tall figure to fit 30 bars
fig, ax = plt.subplots(figsize=(9, 7))
# One colour per bar: orange if beta > 1, blue otherwise
colors = ["#eb6834" if b > 1 else "#2a78d6" for b in plot_betas.values]
# Horizontal bars: ticker labels on the y-axis, beta as bar length
ax.barh(plot_betas.index, plot_betas.values, color=colors, height=0.7)
# Dashed vertical reference line at beta = 1
ax.axvline(1.0, color="#52514e", linewidth=1, linestyle="--")
# Label the reference line near the top of the chart
ax.annotate(
    "market beta = 1",  # the text
    xy=(1.0, len(plot_betas) - 0.3),  # anchor at x = 1, just below the top bar
    xytext=(4, 0),  # nudge 4 points to the right
    textcoords="offset points",  # the nudge is in points
    fontsize=9,  # text size
    color="#52514e",  # dark grey
    va="center",  # vertical centre alignment
)
# Title
ax.set_title("Beta against an equal-weight basket of the 30 sample stocks")
# x-axis label
ax.set_xlabel("Beta")
# Vertical grid lines help read bar lengths
ax.grid(axis="x")
# Horizontal grid lines would clutter the bars, so hide them
ax.grid(axis="y", visible=False)
# enumerate gives bar position y (0, 1, 2, ...) and beta value v
for y, v in enumerate(plot_betas.values):
    # Write the beta, to 2 decimals, just past the end of each bar
    ax.text(v + 0.015, y, f"{v:.2f}", va="center", fontsize=8, color="#52514e")
# Render
plt.show()

### Exercise 3 (worked solution)

Defensive sectors — Utilities, Consumer Staples, Health Care — are supposed
to have betas below 1. Group `beta_table` by sector and check whether that
holds in this sample. Where it does not, can you suggest why? (Hint: this
sample covers 2013–2018, a period with a specific interest-rate story.)

**Approach.** `groupby("sector")` splits the 30 rows by sector; `.agg(...)`
with *named aggregations* (`new_name=(column, function)`) produces one
summary row per sector: number of stocks, mean, lowest and highest beta, and
the share of stocks with beta below 1. The lambda `lambda b: (b < 1).mean()`
turns each beta into True/False and averages, giving a fraction. A list
comprehension then prints the individual names in the three defensive
sectors. Nothing is overwritten: the result goes into a new table,
`sector_betas`.

In [ ]:
# Exercise 3: average beta by sector
# Group the 30 stocks by sector and summarise the covariance beta of each group
sector_betas = beta_table.groupby("sector")["beta_covariance"].agg(
    n_stocks="count",  # how many of our stocks are in the sector
    mean_beta="mean",  # average beta in the sector
    min_beta="min",  # lowest beta in the sector
    max_beta="max",  # highest beta in the sector
    share_below_1=lambda b: (b < 1).mean(),  # fraction of the sector's stocks with beta < 1
).sort_values("mean_beta")  # lowest average beta first

# The three sectors textbooks call defensive
defensive = ["Utilities", "Consumer Staples", "Health Care"]
# Print each defensive stock's individual beta
for sector in defensive:
    # Rows of beta_table in this sector, lowest beta first
    members = beta_table[beta_table["sector"] == sector]["beta_covariance"].sort_values()
    # One line per sector, e.g. "Utilities: DUK 0.53, NEE 0.63"
    print(f"{sector}: " + ", ".join(f"{t} {b:.2f}" for t, b in members.items()))

# Display the sector summary, rounded
sector_betas.round(3)

**Discussion.** In this sample the textbook rule **holds**: every stock in the
three defensive sectors has a beta below 1.

- Utilities have the lowest average beta, about 0.58 (DUK 0.53, NEE 0.63).
- Consumer Staples come next at about 0.71 (KO 0.69, WMT 0.71, PG 0.73).
- Health Care averages about 0.90, the least defensive of the three, and UNH
  at 0.99 is essentially a market-beta stock. UNH is a managed-care insurer
  whose profits were driven by Affordable Care Act enrolment and policy news,
  not the classic "people still buy medicine in a recession" story of the
  drug makers (JNJ 0.82, PFE 0.88).
- At the other end, Financials average about 1.37 and every bank is above 1.2.

**The interest-rate story.** 2013–2018 began with the May 2013 "taper
tantrum", then years of near-zero short rates, the first Fed rate rise in
December 2015, and a sharp rise in yields after the November 2016 election.
Utilities and staples behaved as **bond proxies**: investors bought them for
their dividends, so their prices moved with Treasury yields (falling when
yields jumped) rather than with the equity market. That makes their equity
beta *even lower* than usual — but it does not make them low-risk: DUK still
had a −26% maximum drawdown and a near-zero Sharpe ratio, much of it from rate
moves that a beta against equities cannot see. Banks are the mirror image:
rising rates widen their lending margins, so rate news tends to move all four
together and in the same direction as the broad market's rate-driven rallies,
which pushes their betas up (recall the 0.83 JPM–BAC correlation).

Two cautions. The groups are tiny (one to four stocks each), and the "market"
is our own equal-weight basket, in which the four banks are more than 13% of
the weight, which itself nudges bank betas up. A beta is only as good as the
market proxy and the period it is estimated over.

## 9. Save the outputs

The next cell saves two results to `data/processed/` so later notebooks can
read them without recomputing:

- `processed.mkdir(exist_ok=True)` creates the folder, and does nothing if it
  already exists (without `exist_ok=True` a second run would raise an error).
- `DataFrame.to_csv(path)` writes the table to a comma-separated text file,
  including the index (the dates, or the tickers).

Re-running it simply overwrites the files with identical content. Note that the
exercise solutions above deliberately worked on copies (`stats_2015`,
`stats_calmar`), so the saved `risk_return_stats.csv` has the same four columns
it always had.

In [ ]:
# Path to the processed-data folder
processed = DATA / "processed"
# Create it if missing; exist_ok=True means "no error if it is already there"
processed.mkdir(exist_ok=True)

# Save the full daily simple-return matrix (dates x tickers)
simple_ret.to_csv(processed / "daily_returns.csv")
# Save the five-year risk-return table (one row per ticker)
stats.to_csv(processed / "risk_return_stats.csv")

# Confirm what was written
print("Wrote:")
# Full path of the first file
print("  ", processed / "daily_returns.csv")
# Full path of the second file
print("  ", processed / "risk_return_stats.csv")

---

## Recap

1. Simple returns for portfolio arithmetic; log returns for time-series work.
2. Annualise by 252 for means and $\sqrt{252}$ for volatility — a convention,
   not a law.
3. Volatility clusters. A single number for a five-year period is a summary,
   not a description.
4. Implied volatility sits above realised most of the time; that gap is a
   risk premium, not free money.
5. Correlations rise when you most need them not to.
6. Returns have fat tails. Normal-distribution VaR understates risk.
7. When two routes to the same number disagree, you have a bug.

Everything so far has been **descriptive** — measuring what happened.
Notebook 03 starts building the inputs a **predictive** model needs.

## Your turn

1. Run every cell from the top (**Run All**, or `Shift + Enter` repeatedly)
   and make sure each one finishes without an error.
2. Read the explanation above each cell and the comments in the code, and
   check that your output matches what the text says to look for.
3. Study the three worked exercise solutions and their discussions.
4. Complete the **Canvas comprehension quiz for Notebook 02**. It asks what
   specific lines of code do, how the methods work (returns, annualisation,
   Sharpe, drawdown, correlation, fat tails, VaR, beta), and what the results
   mean.

Then commit and push your run notebook:

```
git add notebooks/02_returns_and_risk.ipynb
git commit -m "Completed notebook 02"
git push
```

Next: `03_features_and_targets.ipynb`